In [1]:
%load_ext autoreload
%autoreload 2
%load_ext jupyter_black

In [2]:
import polars as pl
from tabseq_torch.datasets.names.ds import create_lzdataset, dc
from tabseq_torch.dl import ColumnType, GroupData, dummy_collate_fn, random_split
from torch.utils.data import DataLoader

In [3]:
pl.__version__

'1.44.2'

In [4]:
pl.scan_parquet(dc.num).head(3).collect()

G,name_chars,T,F,F_Rank,vc,index
i64,str,f64,i64,f64,str,i64
0,"""K""",1.0,1028,38.0,"""c""",0
0,"""h""",2.0,6547,60.0,"""c""",1
0,"""o""",3.0,10761,65.0,"""v""",2


In [5]:
pl.scan_parquet(dc.targets).head(3).collect()

anotatin,G,labels
str,i64,u32
"""Arabic.txt""",0,0
"""Arabic.txt""",1,0
"""Arabic.txt""",2,0


In [6]:
lazy_dataset = create_lzdataset()

In [7]:
len(lazy_dataset)

20074

In [8]:
gd: GroupData = next(iter(lazy_dataset))

In [9]:
print(gd)

gid:
 13555 
tb seqs: 
{'name_chars': tensor([14, 54, 37, 38, 47, 44, 48]), 'vc': tensor([2, 3, 2, 3, 2, 2, 3]), 'F': tensor([ 1050,  4578,  2955, 10292,  9341,  5894, 10761]), 'F_Rank': tensor([39., 54., 52., 64., 62., 57., 65.]), 'T': tensor([1., 2., 3., 4., 5., 6., 7.])} 
 target: 14


In [10]:
loader = DataLoader(
    dataset=lazy_dataset,
    batch_size=4,
    collate_fn=dummy_collate_fn,
    num_workers=0,
)

In [11]:
n = 0
for batch in loader:
    for d in batch:
        n += len(d)
print(n)

143622


In [ ]:
train_ds, test_ds = random_split(lazy_dataset, test_size=0.2)
len_train_ds = len(train_ds)
len_test_ds = len(test_ds)
len_lazy_dataset = len(lazy_dataset)
print(len_train_ds)
print(len_test_ds)
print(len_test_ds / len_lazy_dataset)
len_train_ds + len_test_ds == len_lazy_dataset

In [ ]:
embedding_dim = 20
embedding_dims = dict.fromkeys(
    lazy_dataset.get_columns_by_type(column_type=ColumnType.CAT), embedding_dim
)
embedding_dims["vc"] = 10
print(lazy_dataset.embedding_dims_template(embedding_dims))